# Étape 4 — Backbones plus forts (timm)

## 1. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()   # racine du depot (les notebooks sont a la racine)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

from dc_utils.paths import IMAGE_DIR, ARTIFACTS_DIR
from dc_utils.training import TrainConfig, train_model, evaluate, OcclusionDataset, default_eval_tf

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device, '| timm', timm.__version__)

EXP_DIR = ROOT / 'experiments'
EXP_DIR.mkdir(exist_ok=True, parents=True)

df_train = pd.read_csv(ROOT / 'data' / 'train_split.csv')
df_val   = pd.read_csv(ROOT / 'data' / 'val_split.csv')
print('train:', len(df_train), '| val:', len(df_val))

<repo>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device = cuda | timm 1.0.27
train: 89793 | val: 9977


In [2]:
# Régresseur générique : n'importe quel backbone timm + tête linéaire 1 sortie + sigmoid
class TimmRegressor(nn.Module):
    """Backbone timm -> Linear(1) -> sigmoid.

    bias_init=logit(p0) initialise la sortie du sigmoid à p0 (moyenne du train),
    ce qui évite la saturation de la sigmoid en début d'entraînement (sinon la
    tête fraîche produit des logits trop grands, sigmoid saturée à 0, gradient nul,
    le modèle reste bloqué à pred=0).
    """
    def __init__(self, backbone_name: str, pretrained: bool = True, bias_init: float | None = None):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=1)
        if bias_init is not None:
            # timm expose toujours la dernière couche linéaire via get_classifier()
            head = self.backbone.get_classifier()
            with torch.no_grad():
                head.bias.fill_(bias_init)
                head.weight.mul_(0.01)  # weights petits -> sortie initiale ~= bias = logit(p0)
    def forward(self, x):
        return torch.sigmoid(self.backbone(x)).squeeze(-1)

# Précalcul du biais initial = logit(mean_target_train). Vaut ~-2.42 pour notre dataset.
_p0 = float(df_train['FaceOcclusion'].mean())
HEAD_BIAS_INIT = float(np.log(_p0 / (1.0 - _p0)))
print(f'mean target = {_p0:.4f}  ->  bias_init = logit(p0) = {HEAD_BIAS_INIT:.4f}')

def fresh_model(backbone_name: str, seed: int = 42, bias_init: float | None = HEAD_BIAS_INIT):
    torch.manual_seed(seed); np.random.seed(seed)
    m = TimmRegressor(backbone_name, bias_init=bias_init).to(device)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{backbone_name}: {n/1e6:.2f}M params | head bias init = {bias_init}')
    return m

RESULTS: list[dict] = []
def log_run(name, history, best, notes=''):
    history.to_csv(EXP_DIR / f'history_{name}.csv', index=False)
    RESULTS.append({'name': name, 'best_epoch': best['epoch'], 'val_score': best['score'],
                    'err_F': best['err_F'], 'err_M': best['err_M'], 'gap': best['gap'], 'notes': notes})
    print(f'\n>>> {name}: val_score = {best["score"]:.5f} (epoch {best["epoch"]})\n')

mean target = 0.0827  ->  bias_init = logit(p0) = -2.4069


## 2. F1 — efficientnet_b0

EfficientNet-B0 : **5.3M params**, fait pour 224×224 (parfait), pretrain ImageNet. ~2× plus lent que mobilenet_v3_small mais déjà bien plus expressif.

In [3]:
model = fresh_model('efficientnet_b0')

cfg = TrainConfig(epochs=8, lr=5e-4, weight_decay=1e-4, batch_size=128, use_amp=True)
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_F1_effb0.pt', log_prefix='[F1] ',
)
log_run('F1_effb0', hist, best, notes='efficientnet_b0, lr=5e-4, 8ep, BS=128, 3090/AMP, bias-init')

efficientnet_b0: 4.01M params | head bias init = -2.4068707550103765


[F1] ep 1 | loss 0.00215 | score 0.00152 | err_F 0.00087 | err_M 0.00130 | gap 0.00043 | 159s *


[F1] ep 2 | loss 0.00130 | score 0.00144 | err_F 0.00079 | err_M 0.00122 | gap 0.00043 | 151s *


[F1] ep 3 | loss 0.00102 | score 0.00126 | err_F 0.00074 | err_M 0.00109 | gap 0.00034 | 151s *


[F1] ep 4 | loss 0.00073 | score 0.00122 | err_F 0.00067 | err_M 0.00104 | gap 0.00037 | 155s *


[F1] ep 5 | loss 0.00052 | score 0.00121 | err_F 0.00080 | err_M 0.00108 | gap 0.00027 | 155s *


[F1] ep 6 | loss 0.00041 | score 0.00117 | err_F 0.00053 | err_M 0.00096 | gap 0.00043 | 165s *


[F1] ep 7 | loss 0.00031 | score 0.00106 | err_F 0.00051 | err_M 0.00088 | gap 0.00037 | 170s *


[F1] ep 8 | loss 0.00026 | score 0.00114 | err_F 0.00051 | err_M 0.00093 | gap 0.00042 | 164s

>>> F1_effb0: val_score = 0.00106 (epoch 7)



## 3. F2 — convnext_tiny

ConvNeXt-Tiny : **28M params**, ~4-5× plus lent que mobilenet, mais souvent +1-2% sur ImageNet vs efficientnet_b0. Architecture moderne (2022).

**Hyperparams** : `lr=3e-4` (encore plus bas, modèle plus gros), 8 epochs.

⚠️ Si OOM, baisser `batch_size` à 64 ou 96. Temps estimé : ~12 min/epoch → ~1h35.

**Note** : convnext_tiny preferred input is 224×224 ✓.

In [3]:
model = fresh_model('convnext_tiny')
# 3090 : AMP ON + batch_size=64 (28M params, VRAM ok mais on garde marge).
cfg = TrainConfig(epochs=8, lr=1e-4, weight_decay=1e-4, batch_size=64, use_amp=True)
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_F2_convnext.pt', log_prefix='[F2] ',
)
log_run('F2_convnext', hist, best, notes='convnext_tiny, lr=1e-4, 8ep, BS=64, 3090/AMP, bias-init')

convnext_tiny: 27.82M params | head bias init = -2.4068707550103765


[F2] ep 1 | loss 0.00206 | score 0.00188 | err_F 0.00104 | err_M 0.00160 | gap 0.00056 | 182s *


[F2] ep 2 | loss 0.00134 | score 0.00144 | err_F 0.00090 | err_M 0.00126 | gap 0.00036 | 166s *


[F2] ep 3 | loss 0.00103 | score 0.00137 | err_F 0.00063 | err_M 0.00112 | gap 0.00050 | 152s *


[F2] ep 4 | loss 0.00071 | score 0.00130 | err_F 0.00067 | err_M 0.00109 | gap 0.00042 | 151s *


[F2] ep 5 | loss 0.00059 | score 0.00121 | err_F 0.00049 | err_M 0.00097 | gap 0.00049 | 183s *


[F2] ep 6 | loss 0.00042 | score 0.00119 | err_F 0.00049 | err_M 0.00095 | gap 0.00047 | 166s *


[F2] ep 7 | loss 0.00032 | score 0.00116 | err_F 0.00043 | err_M 0.00092 | gap 0.00049 | 155s *


[F2] ep 8 | loss 0.00026 | score 0.00120 | err_F 0.00042 | err_M 0.00094 | gap 0.00052 | 162s

>>> F2_convnext: val_score = 0.00116 (epoch 7)



## 4. Synthèse — comparaison aux baselines

In [4]:
# On ajoute la baseline mobilenet en référence (chargée depuis le CSV de step 3)
step3 = pd.read_csv(EXP_DIR / 'step3_summary.csv')
baseline_row = step3[step3['name']=='baseline'].iloc[0]
ref = {'name':'baseline_mbv3s', 'best_epoch': int(baseline_row['best_epoch']),
       'val_score': baseline_row['val_score'], 'err_F': baseline_row['err_F'],
       'err_M': baseline_row['err_M'], 'gap': baseline_row['gap'],
       'notes':'mobilenet_v3_small (référence)'}

df_results = pd.DataFrame([ref] + RESULTS).sort_values('val_score').reset_index(drop=True)
df_results['delta_vs_baseline'] = df_results['val_score'] - ref['val_score']
df_results['gain_%'] = (-df_results['delta_vs_baseline'] / ref['val_score'] * 100).round(1)
df_results.to_csv(EXP_DIR / 'step4_summary.csv', index=False)
df_results

,name,best_epoch,val_score,err_F,err_M,gap,notes,delta_vs_baseline,gain_%
0,F2_convnext,7,0.001164,0.000434,0.000921,0.000486,"convnext_tiny, lr=1e-4, 8ep, BS=64, 3090/AMP, ...",-0.000057,4.7
1,baseline_mbv3s,7,0.001221,0.000641,0.001028,0.000386,mobilenet_v3_small (référence),0.000000,-0.0


convnext donne un très bon résultat 0.00116 vs 0.00122 (baseline)

## 5. Analyse par bucket du meilleur backbone

In [ ]:
from torch.utils.data import DataLoader
from dc_utils.io import OCC_BINS, OCC_LABELS

best_name = df_results.iloc[0]['name']
ckpt_map = {
    'baseline_mbv3s': None,  # pas dispo ici
    'F1_effb0':       'exp_F1_effb0.pt',
    'F2_convnext':    'exp_F2_convnext.pt',
}
ckpt = ckpt_map.get(best_name)
if ckpt is None:
    print(f'best is {best_name}, déjà analysé en step 3 — rien à refaire')
else:
    backbone_name = 'efficientnet_b0' if 'effb0' in best_name else 'convnext_tiny'
    model = fresh_model(backbone_name)
    state = torch.load(ARTIFACTS_DIR / ckpt, map_location=device)
    model.load_state_dict(state['model'])
    ds_val = OcclusionDataset(df_val, IMAGE_DIR, default_eval_tf(), has_target=True)
    dl_val = DataLoader(ds_val, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    metrics, df_pred = evaluate(model, dl_val, device)
    print('val score:', metrics)

    df_pred['occ_bin'] = pd.cut(df_pred['target'], bins=OCC_BINS, labels=OCC_LABELS, include_lowest=True)
    df_pred['w'] = 1.0/30.0 + df_pred['target']
    df_pred['weighted_sq_err'] = df_pred['w'] * (df_pred['pred'] - df_pred['target'])**2
    tot = df_pred['weighted_sq_err'].sum()
    by_bucket = df_pred.groupby('occ_bin', observed=True).agg(
        n=('target','size'), mean_pred=('pred','mean'), mean_target=('target','mean'),
        weighted_err_share=('weighted_sq_err', lambda s: s.sum()/tot),
    ).round(4)
    by_bucket['bias'] = (by_bucket['mean_pred'] - by_bucket['mean_target']).round(4)
    df_pred.to_csv(EXP_DIR / f'preds_val_{best_name}.csv', index=False)
    print('\nBucket analysis:'); print(by_bucket)

convnext_tiny: 27.82M params | head bias init = -2.4068707550103765
val score: {'score': 0.0011638915761374422, 'err_F': 0.00043441967916024444, 'err_M': 0.000920734277145043, 'gap': 0.00048631459798479857, 'n_F': 3229, 'n_M': 6748}

Bucket analysis:
            n  mean_pred  mean_target  weighted_err_share    bias
occ_bin                                                          
0-5      5009     0.0257       0.0194              0.1001  0.0063
5-10     1873     0.0753       0.0723              0.1003  0.0030
10-20    1950     0.1432       0.1423              0.2630  0.0009
20-30     833     0.2416       0.2430              0.1928 -0.0014
30-50     309     0.3341       0.3414              0.2172 -0.0073
50-70       3     0.3721       0.5045              0.1266 -0.1324
